In [10]:
import requests
import datetime
import time

from utils.ingestion import Ingestion 
from utils.file_utils import FileUtils

In [11]:
ingestion = Ingestion()

configs = ingestion.load_config_json("cities", "metrics")

today = datetime.date.today().isoformat()
today_str = today.replace("-", "")
print(f"Current date: {today}")

Current date: 2026-10-15


In [12]:
base_url = "https://api.open-meteo.com/v1/forecast"

metrics = configs["metrics"]["forecast"]
cities = configs["cities"]

hourly_metrics = metrics.get("hourly")

# Contadores para acompanhar a execução
successful_cities = 0
failed_cities = 0

# Configurações de tentativas
max_attempts = 3
timeout_seconds = 45

for city, coordinates in cities.items():
    latitude = coordinates["latitude"]
    longitude = coordinates["longitude"]

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "hourly": ",".join(hourly_metrics),
        "forecast_days": 7,
        "timezone": "UTC",
    }

    print(f"\nColetando dados de: {city}")

    try:
        # Tentar coletar os dados até 3 vezes
        for attempt in range(1, max_attempts + 1):
            try:
                response = requests.get(
                    base_url,
                    params=params,
                    timeout=timeout_seconds,
                )

                # Repetir apenas em erros HTTP temporários
                retryable_statuses = {429, 500, 502, 503, 504}

                if response.status_code in retryable_statuses:
                    if attempt < max_attempts:
                        wait_seconds = 2 ** attempt

                        print(
                            f"Erro HTTP {response.status_code} para {city}. "
                            f"Tentativa {attempt}/{max_attempts}. "
                            f"Nova tentativa em {wait_seconds}s..."
                        )

                        time.sleep(wait_seconds)
                        continue

                # Identificar respostas HTTP de erro
                response.raise_for_status()

                # Converter a resposta em JSON
                data = response.json()

                # Validar a estrutura básica da resposta
                if "hourly" not in data:
                    raise ValueError(
                        "A resposta da API não contém a chave 'hourly'."
                    )

                if "time" not in data["hourly"]:
                    raise ValueError(
                        "A resposta não contém os horários esperados."
                    )

                # Confirmar que a coleta terminou
                break

            except requests.exceptions.RequestException as e:
                if attempt == max_attempts:
                    raise

                wait_seconds = 2 ** attempt

                print(
                    f"Falha na tentativa {attempt}/{max_attempts} "
                    f"para {city}: {e}"
                )

                print(f"Nova tentativa em {wait_seconds}s...")
                time.sleep(wait_seconds)

        # Salvar somente depois de uma resposta válida
        output_path = (
            f"../data/bronze/open_meteo/forecast/"
            f"{today_str}/{city}_{today_str}.json"
        )

        FileUtils.write_json(data, output_path)

        successful_cities += 1

        print("Coleta concluída com sucesso!")
        print(
            "Horários coletados:",
            len(data["hourly"]["time"]),
        )
        print("Arquivo salvo em:", output_path)

    except requests.exceptions.RequestException as e:
        failed_cities += 1
        print(f"Erro na requisição para {city}: {e}")

    except (ValueError, KeyError) as e:
        failed_cities += 1
        print(f"Erro nos dados recebidos para {city}: {e}")

    except Exception as e:
        failed_cities += 1
        print(f"Erro inesperado para {city}: {e}")

print("\n========== RESUMO DA COLETA ==========")
print("Cidades processadas:", len(cities))
print("Coletas bem-sucedidas:", successful_cities)
print("Coletas com falha:", failed_cities)


Coletando dados de: veghel
Coleta concluída com sucesso!
Horários coletados: 168
Arquivo salvo em: ../data/bronze/open_meteo/forecast/20261015/veghel_20261015.json

Coletando dados de: sao_tiago
Coleta concluída com sucesso!
Horários coletados: 168
Arquivo salvo em: ../data/bronze/open_meteo/forecast/20261015/sao_tiago_20261015.json

Coletando dados de: amsterdam
Coleta concluída com sucesso!
Horários coletados: 168
Arquivo salvo em: ../data/bronze/open_meteo/forecast/20261015/amsterdam_20261015.json

Coletando dados de: berlin
Coleta concluída com sucesso!
Horários coletados: 168
Arquivo salvo em: ../data/bronze/open_meteo/forecast/20261015/berlin_20261015.json

Coletando dados de: paris
Coleta concluída com sucesso!
Horários coletados: 168
Arquivo salvo em: ../data/bronze/open_meteo/forecast/20261015/paris_20261015.json

Coletando dados de: rome
Coleta concluída com sucesso!
Horários coletados: 168
Arquivo salvo em: ../data/bronze/open_meteo/forecast/20261015/rome_20261015.json

Col

In [13]:
from pathlib import Path
import json

bronze_dir = (
    Path.cwd().parent
    / "data"
    / "bronze"
    / "open_meteo"
    / "forecast"
    / today_str
)

json_files = sorted(bronze_dir.glob("*.json"))

print("Diretório:", bronze_dir)
print("Arquivos JSON encontrados:", len(json_files))

for path in json_files:
    try:
        with path.open("r", encoding="utf-8") as file:
            data = json.load(file)

        hourly = data.get("hourly", {})
        times = hourly.get("time", [])

        print(
            f"{path.name}: "
            f"{len(times)} horários | "
            f"{'OK' if len(times) == 168 else 'VERIFICAR'}"
        )

    except (json.JSONDecodeError, OSError) as e:
        print(f"{path.name}: ERRO - {e}")

Diretório: /home/mariarita-resende/Projetos/open_meteo_ingestion/data/bronze/open_meteo/forecast/20261015
Arquivos JSON encontrados: 14
amsterdam_20261015.json: 168 horários | OK
barcelona_20261015.json: 168 horários | OK
berlin_20261015.json: 168 horários | OK
brussels_20261015.json: 168 horários | OK
budapest_20261015.json: 168 horários | OK
copenhagen_20261015.json: 168 horários | OK
dublin_20261015.json: 168 horários | OK
lisbon_20261015.json: 168 horários | OK
paris_20261015.json: 168 horários | OK
prague_20261015.json: 168 horários | OK
rome_20261015.json: 168 horários | OK
sao_tiago_20261015.json: 168 horários | OK
veghel_20261015.json: 168 horários | OK
vienna_20261015.json: 168 horários | OK
